## Import Library

In [ ]:
import pandas as pd
import numpy as np
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


**Load Dataset**

In [ ]:
reviews = pd.read_excel('./Popchip_Reviews.xlsx')
reviews.head(2)

,Id,UserId,Rating,Priority,Title,Text
0,23689,A21SYVGVNG8RAS,5,Low,Yummy snacks!,Popchips are the bomb!! I use the parmesan ga...
1,23690,AQJYXC0MPRQJL,5,Low,Great chip that is different from the rest,I like the puffed nature of this chip that mak...


**Melihat Struktur Data**

In [ ]:
reviews.shape

(564, 6)

In [ ]:
reviews.Priority.value_counts()

,count
Priority,
Low,447
High,117


**Load Model NLP (spaCy)**

In [ ]:
nlp = spacy.load("en_core_web_sm")

## Preprocessing Text

**Lowercase & Cleaning**

In [ ]:
def lower_remove(series):
    output = series.str.lower()
    output = output.str.replace(r'\[.*?\]', '', regex=True)   # ✅ fix
    output = output.str.replace(r'[^\w\s]', '', regex=True)   # ✅ fix sebelumnya
    return output


Tokenizing + Lemmatization + Stopword Removal

In [ ]:
def token_lemma_non_stop_word(text):
  doc = nlp(text)
  output = [token.lemma_ for token in doc if not token.is_stop]
  return ' '.join(output)

Gabungkan Semua Preprocessing

In [ ]:
def clean_and_normalize(series):
  output = lower_remove(series)
  output = output.apply(token_lemma_non_stop_word)
  return output

Terapkan hasil pre-processing ke dataset

Membuat kolom baru cleaned_text


Berisi teks yang sudah bersih dan siap diproses

In [ ]:
reviews['cleaned_text'] = clean_and_normalize(reviews['Text'])
reviews['cleaned_text']

,cleaned_text
0,popchip bomb use parmesan garlic scoop cotta...
1,like puff nature chip make unique chip market ...
2,love chip big fan potato chip not discover p...
3,taste like potatoe stix get grade school lunch...
4,chip great look like flattened rice cake tas...
...,...
559,love potato chip eat bagful thank power prov...
560,popchip hard find order case amazon regular ba...
561,healthy alternative chip taste great great c...
562,good ve start get automatically like origina...


**Feature Extraction dengan TF-IDF**

Mengubah teks menjadi numerik menggunakan TF-IDF
Parameter penting:
min_df=0.05 → kata muncul minimal 5% pada dokumen


max_df=0.2 → buang kata terlalu sering (kurang informatif)

In [ ]:
tv = TfidfVectorizer(stop_words='english', min_df=0.05, max_df=.2)
Xt = tv.fit_transform(reviews['cleaned_text'])
Xt_df = pd.DataFrame(Xt.toarray(), columns=tv.get_feature_names_out())
Xt_df

,100,alternative,amazon,bad,bake,baked,bbq,big,bit,box,...,thing,think,time,variety,ve,vinegar,want,way,weight,work
0,0.0,0.465515,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.348295,0.193511,0.000000,0.000000,0.000000
2,0.0,0.000000,0.354088,0.000000,0.0,0.000000,0.000000,0.428869,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.354475,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
559,0.0,0.324462,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.299888,0.0,0.000000,0.000000,0.000000,0.337388,0.657147,0.000000
560,0.0,0.000000,0.190702,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.197896,0.000000,0.208227,0.000000,0.000000,0.247474
561,0.0,0.378621,0.000000,0.380993,0.0,0.396437,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
562,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.000000,0.0,0.543142,0.000000,0.000000,0.000000,0.000000,0.000000


Topic Modeling dengan model NMF (unsupervised learning)

In [ ]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=10, random_state=42, max_iter=500)
W = nmf.fit_transform(Xt_df) # documents-topics
H = nmf.components_ # topics-terms

H → hubungan topik dengan kata

Menunjukkan jumlah:

topik -> terdapat 5 topik di dokumen

terdapat 81 kata dalam setiap topik

In [ ]:
H.shape

(10, 81)

In [ ]:
H[0][:20]

array([0.1300751 , 0.        , 0.0158107 , 0.44279504, 0.46688753,
       0.07035021, 0.        , 0.32369247, 0.54991929, 0.46240876,
       4.52997549, 0.11789091, 0.        , 0.41424905, 0.        ,
       0.        , 0.        , 0.36517905, 0.01944818, 0.        ])

**Menampilkan Top Words tiap Topik**

Mengambil kata dengan bobot tertinggi


Menampilkan kata kunci tiap topik

In [ ]:
def display_topics(H, num_words=10):
    for topic_num, topic_array in enumerate(H):
        top_features = topic_array.argsort()[::-1][:num_words]
        top_words = [tv.get_feature_names_out()[i] for i in top_features]
        print("Topic", topic_num+1, ":", ', '.join(top_words))

In [ ]:
display_topics(H)

Topic 1 : br, lime, thing, think, review, little, say, bit, texture, expect
Topic 2 : sweet, salty, light, rice, texture, little, think, crunchy, greasy, right
Topic 3 : healthy, alternative, delicious, enjoy, feel, work, nice, look, case, think
Topic 4 : bbq, want, size, light, onion, favorite, price, little, small, cheddar
Topic 5 : fat, low, crunch, satisfy, price, high, enjoy, recommend, strong, need
Topic 6 : order, amazon, case, time, store, box, thing, price, know, 100
Topic 7 : vinegar, favorite, pepper, sea, original, garlic, cheddar, sour, onion, definitely
Topic 8 : weight, product, food, favorite, diet, baked, want, salty, definitely, low
Topic 9 : regular, pop, ve, alternative, lot, crunchy, baked, tasty, greasy, definitely
Topic 10 : pack, serve, variety, single, food, lot, tasty, diet, 100, need


Representasi Dokumen ke Topik

In [ ]:
doc_topics = pd.DataFrame(W)
doc_topics.columns = ['orders', 'taste & texture', 'Healthy Alternative', 'Flavor Variants', 'Diet & Low-fat']
doc_topics

,orders,taste & texture,Healthy Alternative,Flavor Variants,Diet & Low-fat
0,0.000000,0.000000,0.403012,0.000000,0.000000
1,0.055080,0.000000,0.023755,0.115179,0.088048
2,0.067787,0.000000,0.000000,0.000000,0.153890
3,0.017647,0.002463,0.000000,0.000000,0.029204
4,0.000000,0.016166,0.040860,0.044669,0.190659
...,...,...,...,...,...
559,0.025953,0.010370,0.050308,0.000000,0.168847
560,0.108660,0.000000,0.022080,0.157261,0.032282
561,0.084727,0.000000,0.200482,0.000000,0.091203
562,0.019073,0.000000,0.000000,0.085505,0.037631


Gabungkan dengan Data Asli

In [ ]:
reviews_topics = pd.concat([reviews.Text, doc_topics], axis=1)
reviews_topics.head()

,Text,orders,taste & texture,good,flavor,health
0,Popchips are the bomb!! I use the parmesan ga...,0.000000,0.000000,0.403012,0.000000,0.000000
1,I like the puffed nature of this chip that mak...,0.055080,0.000000,0.023755,0.115179,0.088048
2,I just love these chips! I was always a big f...,0.067787,0.000000,0.000000,0.000000,0.153890
3,"These tasted like potatoe stix, that we got in...",0.017647,0.002463,0.000000,0.000000,0.029204
4,These chips are great! They look almost like ...,0.000000,0.016166,0.040860,0.044669,0.190659
